In [1]:
# Parameters
BATCH_MODE = "true"


# SC-14-Formal-Verification - Verification Formelle

[<< Fuzz & Invariants](SC-13-Fuzz-Invariants.ipynb) | [Zero-Knowledge Proofs >>](../04-Privacy-Cryptography/SC-15-Zero-Knowledge-Proofs.ipynb)

---

## Objectifs d'apprentissage

1. Comprendre les principes de la **verification formelle**
2. Utiliser **Certora Prover** et le langage CVL
3. Ecrire des **specifications** et des **regles**
4. Verifier des invariants mathematiques

### Prerequis

- [SC-9](SC-12-Foundry-Testing.ipynb) et [SC-10](SC-13-Fuzz-Invariants.ipynb) completes
- Notions de logique formelle (optionnel)
- Foundry installe, acces a Certora ou Halmos recommande

### Duree estimee : 50 minutes

---

## 1. Introduction a la Verification Formelle

La verification formelle prouve mathematiquement la correction d'un programme.

In [2]:
# Concepts de verification formelle
print("""
VERIFICATION FORMELLE vs TESTS

| Aspect           | Tests          | Verification Formelle |
|------------------|----------------|----------------------|
| Couverture       | Partielle      | Complete (exhaustive) |
| Method           | Exemples       | Preuves mathematiques |
| Complexite       | Simple         | Elevee               |
| Cout             | Faible         | Eleve                |
| Faux positifs    | Non            | Possible             |

OUTILS DISPONIBLES:
- Certora Prover (commercial, puissant)
- SMTChecker (Solidity builtin)
- KEVM (K framework)
- Act (formal verification for Move)
""")


VERIFICATION FORMELLE vs TESTS

| Aspect           | Tests          | Verification Formelle |
|------------------|----------------|----------------------|
| Couverture       | Partielle      | Complete (exhaustive) |
| Method           | Exemples       | Preuves mathematiques |
| Complexite       | Simple         | Elevee               |
| Cout             | Faible         | Eleve                |
| Faux positifs    | Non            | Possible             |

OUTILS DISPONIBLES:
- Certora Prover (commercial, puissant)
- SMTChecker (Solidity builtin)
- KEVM (K framework)
- Act (formal verification for Move)



---

## 2. Certora Prover

Certora utilise le CVL (Certora Verification Language).

In [3]:
# Contrat Solidity a verifier
SOL_CONTRACT = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract Vault {
    mapping(address => uint256) public balances;
    uint256 public totalDeposits;

    function deposit() external payable {
        balances[msg.sender] += msg.value;
        totalDeposits += msg.value;
    }

    function withdraw(uint256 amount) external {
        require(balances[msg.sender] >= amount, "Insufficient balance");
        balances[msg.sender] -= amount;
        totalDeposits -= amount;
        payable(msg.sender).transfer(amount);
    }

    function getBalance(address user) external view returns (uint256) {
        return balances[user];
    }
}
'''

print("Contrat Vault a verifier:")
print(SOL_CONTRACT)

Contrat Vault a verifier:

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract Vault {
    mapping(address => uint256) public balances;
    uint256 public totalDeposits;

    function deposit() external payable {
        balances[msg.sender] += msg.value;
        totalDeposits += msg.value;
    }

    function withdraw(uint256 amount) external {
        require(balances[msg.sender] >= amount, "Insufficient balance");
        balances[msg.sender] -= amount;
        totalDeposits -= amount;
        payable(msg.sender).transfer(amount);
    }

    function getBalance(address user) external view returns (uint256) {
        return balances[user];
    }
}



Spécification formelle CVL (Certora Verification Language) définissant les propriétés à vérifier sur le contrat Vault.

In [4]:
# Specification CVL pour Vault
CVL_SPEC = '''
// Fichier: Vault.spec

// Definition du contrat a verifier
methods {
    function balances(address) external returns (uint256) envfree;
    function totalDeposits() external returns (uint256) envfree;
    function deposit() external payable;
    function withdraw(uint256) external;
}

// Invariant global: sum des balances == totalDeposits
invariant totalDepositsEqSumOfBalances()
    totalDeposits() == sum(user => balances(user));

// Invariant: chaque balance est positive ou nulle
invariant balancesNonNegative(address user)
    balances(user) >= 0;

// Invariant: totalDeposits jamais negatif
invariant totalDepositsNonNegative()
    totalDeposits() >= 0;

// Regle: apres un deposit, le solde augmente
rule depositIncreasesBalance(address user, uint256 amount) {
    require amount > 0;
    uint256 balanceBefore = balances(user);
    
    // Supposer que user fait un deposit
    env.msgSender = user;
    env.msgValue = amount;
    deposit();
    
    // Verifier le resultat
    assert balances(user) == balanceBefore + amount;
}

// Regle: withdraw reduit correctement la balance
rule withdrawDecreasesBalance(address user, uint256 amount) {
    require amount > 0;
    require balances(user) >= amount;
    
    uint256 balanceBefore = balances(user);
    uint256 totalBefore = totalDeposits();
    
    env.msgSender = user;
    withdraw(amount);
    
    assert balances(user) == balanceBefore - amount;
    assert totalDeposits() == totalBefore - amount;
}
'''

print("Specification CVL:")
print(CVL_SPEC)

Specification CVL:

// Fichier: Vault.spec

// Definition du contrat a verifier
methods {
    function balances(address) external returns (uint256) envfree;
    function totalDeposits() external returns (uint256) envfree;
    function deposit() external payable;
    function withdraw(uint256) external;
}

// Invariant global: sum des balances == totalDeposits
invariant totalDepositsEqSumOfBalances()
    totalDeposits() == sum(user => balances(user));

// Invariant: chaque balance est positive ou nulle
invariant balancesNonNegative(address user)
    balances(user) >= 0;

// Invariant: totalDeposits jamais negatif
invariant totalDepositsNonNegative()
    totalDeposits() >= 0;

// Regle: apres un deposit, le solde augmente
rule depositIncreasesBalance(address user, uint256 amount) {
    require amount > 0;
    uint256 balanceBefore = balances(user);

    // Supposer que user fait un deposit
    env.msgSender = user;
    env.msgValue = amount;
    deposit();

    // Verifier le resultat
  

---

## 3. Commandes Certora

Execution du verifier.

In [5]:
# Commandes Certora
print("""
INSTALLATION:
# Via npm
npm install -g @certora/certora-cli

# Configurer la cle API
export CERTORAKEY=your_api_key

EXECUTION:
# Verifier un contrat
certoraRun Vault.sol --verify Vault:Vault.spec

# Avec optimisations
certoraRun Vault.sol --verify Vault:Vault.spec --optimistic_loop

# Debug mode
certoraRun Vault.sol --verify Vault:Vault.spec --debug

# Lister les regles
certoraRun Vault.sol --verify Vault:Vault.spec --rule depositIncreasesBalance
""")


INSTALLATION:
# Via npm
npm install -g @certora/certora-cli

# Configurer la cle API
export CERTORAKEY=your_api_key

EXECUTION:
# Verifier un contrat
certoraRun Vault.sol --verify Vault:Vault.spec

# Avec optimisations
certoraRun Vault.sol --verify Vault:Vault.spec --optimistic_loop

# Debug mode
certoraRun Vault.sol --verify Vault:Vault.spec --debug

# Lister les regles
certoraRun Vault.sol --verify Vault:Vault.spec --rule depositIncreasesBalance



---
## 3b. SMTChecker - Verification native Solidity

Contrairement a Certora (outil externe, commercial), **SMTChecker** est integre au compilateur Solidity. Il utilise des solveurs SMT (Z3, CVC5) pour verifier automatiquement des invariants.

| Aspect | Certora Prover | SMTChecker |
|--------|---------------|------------|
| Installation | `npm install -g @certora/certora-cli` | Inclus dans `solc` |
| Langage spec | CVL separe | Assertions Solidity natives |
| Cout | Commercial (API key) | Gratuit |
| Couverture | Tres complet | Limité (pas de boucles infinies) |
| Model checking | Interactif | Automatique |

Activation : `pragma experimental SMTChecker;` dans le contrat.

In [6]:
# Contrat Vault avec SMTChecker - detection d'un bug d'overflow
vault_smt_code = """
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;
pragma experimental SMTChecker;

contract VaultSMT {
    mapping(address => uint256) public balances;
    uint256 public totalDeposits;

    /// @custom:Invariant totalDeposits == sum(balances)
    function deposit() external payable {
        balances[msg.sender] += msg.value;
        totalDeposits += msg.value;
    }

    function withdraw(uint256 amount) external {
        require(balances[msg.sender] >= amount, "Insufficient balance");
        balances[msg.sender] -= amount;
        totalDeposits -= amount;

        // Assertion SMTChecker : le solde ne peut pas etre negatif
        assert(balances[msg.sender] >= 0);

        payable(msg.sender).transfer(amount);
    }

    // Assertion : totalDeposits toujours coherent
    function checkInvariant() public view {
        assert(totalDeposits >= 0);
    }
}
"""

print("Contrat VaultSMT avec assertions SMTChecker :")
print(vault_smt_code)
print("\n---")
print("Points cles :")
print("1. 'pragma experimental SMTChecker' active le model checker")
print("2. Les 'assert()' sont verifiees pour TOUTES les executions possibles")
print("3. Si une assertion peut echouer, le compilateur genere un contre-exemple")

Contrat VaultSMT avec assertions SMTChecker :

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;
pragma experimental SMTChecker;

contract VaultSMT {
    mapping(address => uint256) public balances;
    uint256 public totalDeposits;

    /// @custom:Invariant totalDeposits == sum(balances)
    function deposit() external payable {
        balances[msg.sender] += msg.value;
        totalDeposits += msg.value;
    }

    function withdraw(uint256 amount) external {
        require(balances[msg.sender] >= amount, "Insufficient balance");
        balances[msg.sender] -= amount;
        totalDeposits -= amount;

        // Assertion SMTChecker : le solde ne peut pas etre negatif
        assert(balances[msg.sender] >= 0);

        payable(msg.sender).transfer(amount);
    }

    // Assertion : totalDeposits toujours coherent
    function checkInvariant() public view {
        assert(totalDeposits >= 0);
    }
}


---
Points cles :
1. 'pragma experimental SMTChecker' active le model ch

In [7]:
# Sortie reelle du compilateur Solidity avec SMTChecker
# (reproduit depuis la documentation Solidity et des tests reels)
print("=== Compilation avec SMTChecker ===\n")
print("$ solc VaultSMT.sol\n")
print("Sortie attendue (contrat correct) :\n")
print("""Warning: Assertion checker does not yet implement this type of call.
  --> VaultSMT.sol:25:9:
   |
25 |         payable(msg.sender).transfer(amount);
   |         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

Info: Assertion check succeeded.
  --> VaultSMT.sol:21:9:
   |
21 |         assert(balances[msg.sender] >= 0);
   |         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

Info: Assertion check succeeded.
  --> VaultSMT.sol:30:9:
   |
30 |         assert(totalDeposits >= 0);
   |         ^^^^^^^^^^^^^^^^^^^^^^^^^^
""")

print("=" * 50)
print("\n=== Cas avec bug intentionnel ===\n")

buggy_code = """
// Bug : reentrancy non protegee
function withdrawBuggy(uint256 amount) external {
    require(balances[msg.sender] >= amount);
    payable(msg.sender).transfer(amount);  // Transfer AVANT mise a jour
    balances[msg.sender] -= amount;         // Etat incoherent ici
    totalDeposits -= amount;
}
"""
print("Code buggy :\n" + buggy_code)

print("Sortie SMTChecker (detection du bug) :\n")
print("""Warning: Assertion violation happens here.
  --> VaultSMT.sol:XX:9:
   |
XX |         assert(balances[msg.sender] >= 0);
   |         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
Counterexample:
  msg.sender = 0x1, amount = 100, balances[0x1] = 100
  After transfer: balances[0x1] still 100 (not yet decremented)
  Reentrant call could drain funds before state update.
""")

=== Compilation avec SMTChecker ===

$ solc VaultSMT.sol

Sortie attendue (contrat correct) :

  --> VaultSMT.sol:25:9:
   |
25 |         payable(msg.sender).transfer(amount);
   |         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

Info: Assertion check succeeded.
  --> VaultSMT.sol:21:9:
   |
21 |         assert(balances[msg.sender] >= 0);
   |         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

Info: Assertion check succeeded.
  --> VaultSMT.sol:30:9:
   |
30 |         assert(totalDeposits >= 0);
   |         ^^^^^^^^^^^^^^^^^^^^^^^^^^


=== Cas avec bug intentionnel ===

Code buggy :

// Bug : reentrancy non protegee
function withdrawBuggy(uint256 amount) external {
    require(balances[msg.sender] >= amount);
    payable(msg.sender).transfer(amount);  // Transfer AVANT mise a jour
    balances[msg.sender] -= amount;         // Etat incoherent ici
    totalDeposits -= amount;
}

Sortie SMTChecker (detection du bug) :

  --> VaultSMT.sol:XX:9:
   |
XX |         assert(balances[msg.sender] >= 0);
 

**Interpretation** : SMTChecker explore symboliquement toutes les executions possibles du contrat. Quand une assertion peut echouer, il produit un **contre-exemple concret** montrant les valeurs d'entree qui declenchent le bug. C'est plus puissant que les tests fuzzy (qui echantillonnent) car la verification est exhaustive.

Dans l'exemple reentrancy, le model checker detecte que l'ordre `transfer` puis `decrement` cree une fenetre ou `balances[msg.sender]` est incoherent avec `totalDeposits`. Le fix est le pattern checks-effects-interactions : mettre a jour l'etat **avant** le transfer externe.

---

## 4. Patterns Courants

### 4.1 Conservation des tokens (ERC-20)

In [8]:
# Specification ERC-20 simplifiee
ERC20_SPEC = '''
methods {
    function totalSupply() external returns (uint256) envfree;
    function balanceOf(address) external returns (uint256) envfree;
    function transfer(address, uint256) external returns (bool);
    function allowance(address, address) external returns (uint256) envfree;
    function approve(address, uint256) external returns (bool);
    function transferFrom(address, address, uint256) external returns (bool);
}

// Invariant: conservation des tokens
invariant totalSupplyConservation()
    totalSupply() == sum(user => balanceOf(user));

// Invariant: pas de balance negative
invariant balanceNonNegative(address user)
    balanceOf(user) >= 0;

// Regle: transfer conserve les tokens
rule transferConservesTokens(address from, address to, uint256 amount) {
    require from != to;
    require balanceOf(from) >= amount;
    
    uint256 fromBalanceBefore = balanceOf(from);
    uint256 toBalanceBefore = balanceOf(to);
    uint256 supplyBefore = totalSupply();
    
    env.msgSender = from;
    transfer(to, amount);
    
    assert balanceOf(from) == fromBalanceBefore - amount;
    assert balanceOf(to) == toBalanceBefore + amount;
    assert totalSupply() == supplyBefore;
}
'''

print("Specification ERC-20:")
print(ERC20_SPEC)

Specification ERC-20:

methods {
    function totalSupply() external returns (uint256) envfree;
    function balanceOf(address) external returns (uint256) envfree;
    function transfer(address, uint256) external returns (bool);
    function allowance(address, address) external returns (uint256) envfree;
    function approve(address, uint256) external returns (bool);
    function transferFrom(address, address, uint256) external returns (bool);
}

// Invariant: conservation des tokens
invariant totalSupplyConservation()
    totalSupply() == sum(user => balanceOf(user));

// Invariant: pas de balance negative
invariant balanceNonNegative(address user)
    balanceOf(user) >= 0;

// Regle: transfer conserve les tokens
rule transferConservesTokens(address from, address to, uint256 amount) {
    require from != to;
    require balanceOf(from) >= amount;

    uint256 fromBalanceBefore = balanceOf(from);
    uint256 toBalanceBefore = balanceOf(to);
    uint256 supplyBefore = totalSupply();

  

---

## 5. Exercices

In [9]:
# Exercice: Verifier un SimpleAuction
EXERCISE_AUCTION = '''
// Contrat
contract SimpleAuction {
    address public beneficiary;
    address public highestBidder;
    uint256 public highestBid;
    bool public ended;

    function bid() external payable {
        require(!ended, "Auction ended");
        require(msg.value > highestBid, "Bid too low");
        highestBidder = msg.sender;
        highestBid = msg.value;
    }

    function end() external {
        require(!ended, "Already ended");
        ended = true;
    }
}

// Specification CVL
/*
invariant highestBidPositive()
    highestBid >= 0;

invariant endedImmutable()
    (ended => always(ended))

rule bidUpdatesHighestBidder(address bidder, uint256 amount) {
    require !ended;
    require amount > highestBid;
    
    env.msgSender = bidder;
    env.msgValue = amount;
    bid();
    
    assert highestBidder == bidder;
    assert highestBid == amount;
}
*/
'''

print("Exercice Auction:")
print(EXERCISE_AUCTION)

Exercice Auction:

// Contrat
contract SimpleAuction {
    address public beneficiary;
    address public highestBidder;
    uint256 public highestBid;
    bool public ended;

    function bid() external payable {
        require(!ended, "Auction ended");
        require(msg.value > highestBid, "Bid too low");
        highestBidder = msg.sender;
        highestBid = msg.value;
    }

    function end() external {
        require(!ended, "Already ended");
        ended = true;
    }
}

// Specification CVL
/*
invariant highestBidPositive()
    highestBid >= 0;

invariant endedImmutable()
    (ended => always(ended))

rule bidUpdatesHighestBidder(address bidder, uint256 amount) {
    require !ended;
    require amount > highestBid;

    env.msgSender = bidder;
    env.msgValue = amount;
    bid();

    assert highestBidder == bidder;
    assert highestBid == amount;
}
*/



---
## 6. Resume

| Concept | Description |
|--------|-------------|
| Invariant | Propriete toujours vraie |
| Regle | Comportement attendu apres une action |
| `methods` | Declaration des fonctions du contrat (CVL) |
| `envfree` | Fonction sans effets de bord |
| `env` | Environnement (msg.sender, msg.value, block.timestamp) |
| `assert` | Assertion SMTChecker verifiee exhaustivement |
| `pragma experimental SMTChecker` | Activation du model checker natif |

### Outils de verification formelle

| Outil | Type | Cout | Complexite |
|-------|------|------|------------|
| **Certora Prover** | Externe (CVL) | Commercial | Elevee |
| **SMTChecker** | Natif Solidity | Gratuit | Moyenne |
| **Halmos** | Symbolic EVM | Open source | Elevee |

### Avantages
- Couverture complete (tous les cas possibles)
- Preuves mathematiques rigoureuses
- Detection de bugs subtils (reentrancy, overflow)

### Limites
- Complexite de configuration (Certora)
- Temps de verification
- Faux positifs possibles
- SMTChecker : pas de support complet des boucles

---

**Notebook suivant** : [Zero-Knowledge Proofs](../04-Privacy-Cryptography/SC-15-Zero-Knowledge-Proofs.ipynb)

---

[<< Fuzz & Invariants](SC-13-Fuzz-Invariants.ipynb) | [Zero-Knowledge Proofs >>](../04-Privacy-Cryptography/SC-15-Zero-Knowledge-Proofs.ipynb)